##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [8]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [9]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")

x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")

print("x_train:", x_train.shape, "y_train:", y_train.shape)
print("x_test :", x_test.shape,  "y_test :", y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
x_train: (50000, 32, 32, 3) y_train: (50000,)
x_test : (10000, 32, 32, 3) y_test : (10000,)


In [10]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")

In [11]:
mobilenet_base = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
mobilenet_base.trainable = False

mobilenet_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Resizing(224, 224),
    layers.Lambda(preprocess_input),
    mobilenet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10)  # logits
], name="cifar10_mobilenetv2")

mobilenet_model.summary()

Model: "cifar10_mobilenetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing_1 (Resizing)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [12]:
mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_mob = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

test_loss_mob, test_acc_mob = mobilenet_model.evaluate(x_test, y_test, verbose=0)
print("MobileNetV2 (frozen) test accuracy:", test_acc_mob)
print("MobileNetV2 (frozen) test loss:", test_loss_mob)

Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 86s 111ms/step - accuracy: 0.5844 - loss: 1.1994 - val_accuracy: 0.8024 - val_loss: 0.5704
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 76s 108ms/step - accuracy: 0.7409 - loss: 0.7438 - val_accuracy: 0.8302 - val_loss: 0.4938
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 76s 107ms/step - accuracy: 0.7582 - loss: 0.6930 - val_accuracy: 0.8280 - val_loss: 0.5068
MobileNetV2 (frozen) test accuracy: 0.8144999742507935
MobileNetV2 (frozen) test loss: 0.539406418800354


In [13]:
mobilenet_base.trainable = True

for layer in mobilenet_base.layers[:-30]:
    layer.trainable = False

print("Trainable layers in backbone:",
      sum(l.trainable for l in mobilenet_base.layers), "/", len(mobilenet_base.layers))

mobilenet_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_mob_ft = mobilenet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

test_loss_mob_ft, test_acc_mob_ft = mobilenet_model.evaluate(x_test, y_test, verbose=0)
print("MobileNetV2 (fine-tuned) test accuracy:", test_acc_mob_ft)
print("MobileNetV2 (fine-tuned) test loss:", test_loss_mob_ft)

Trainable layers in backbone: 30 / 154
Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 108s 141ms/step - accuracy: 0.6663 - loss: 0.9689 - val_accuracy: 0.8264 - val_loss: 0.5099
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 97s 138ms/step - accuracy: 0.7687 - loss: 0.6614 - val_accuracy: 0.8418 - val_loss: 0.4521
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 97s 138ms/step - accuracy: 0.7885 - loss: 0.6022 - val_accuracy: 0.8516 - val_loss: 0.4158
MobileNetV2 (fine-tuned) test accuracy: 0.8489999771118164
MobileNetV2 (fine-tuned) test loss: 0.43188998103141785


- Which model achieved the highest accuracy?

When comparing models based on their accuracy on CIFAR-10, the ResNet50V2 model had the best performance with around 91.6% accuracy after fine-tuning. This result is expected due to ResNet's depth and use of residual connections.


- Which model trained faster?

The custom CNN was the fastest to train because it has fewer layers and parameters, does not use a large pretrained backbone.
The training speed of MobileNetV2 came in between the custom CNN and ResNet50V2; however, ResNet50V2 took the longest time to complete training due to its depth and large number of parameters.

- How might the architecture explain the differences?

Model architecture explains different performance outcomes.
The custom-trained CNN was shallow when it was built, and thus built on the more limited capabilities of those constructing their own versus using an already-existing model based upon the features contained within a sufficiently large, existing training dataset; therefore, it is unable to generalise confidently between those training sample relationships that it does not have representation of.
Pre-trained on a dataset with 1000 images (ImageNet), the MobileNet V2 architecture makes significant use of depthwise separable convolution layers and incorporates an additional layer between these convolution layers for assisting in feature extraction while maintaining computational efficiency; therefore, as a result of having less than half the number of parameters associated with ResNet, MobileNet V2 can achieve comparable levels of accuracy with significantly decreased computation times.
ResNet50V2 is a much deeper network that uses residual connections, enabling very deep feature learning without vanishing gradients. Because of this, it captures more complex visual patterns, leading to the highest accuracy, although at the cost of longer training time and higher computational demand.
